<a href="https://colab.research.google.com/github/anisha-dev26/AI_LAB-1BF24CS054-/blob/main/1BF24CS054_Week8_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# WEEK-8 AI LAB
# First Order Logic (FOL) using Resolution
# Query: Prove that John likes peanuts.

# -----------------------------------------
# STEP 1: INITIALIZE KNOWLEDGE BASE IN CNF
# -----------------------------------------

# Literal format: (positive, predicate, arguments)
# True = Positive literal
# False = Negative literal

KB = [

    # John likes all kinds of food
    [(False, "Food", ("?x",)),
     (True, "Likes", ("John", "?x"))],

    # Apple is food
    [(True, "Food", ("Apple",))],

    # Vegetables are food
    [(True, "Food", ("Vegetables",))],

    # Anyone who eats something and is not killed,
    # that thing is food
    [(False, "Eats", ("?x", "?y")),
     (False, "NotKilled", ("?x",)),
     (True, "Food", ("?y",))],

    # Anil eats peanuts
    [(True, "Eats", ("Anil", "Peanuts"))],

    # Anil is alive
    [(True, "Alive", ("Anil",))],

    # Harry eats everything Anil eats
    [(False, "Eats", ("Anil", "?x")),
     (True, "Eats", ("Harry", "?x"))],

    # Alive implies not killed
    [(False, "Alive", ("?x",)),
     (True, "NotKilled", ("?x",))],

    # Not killed implies alive
    [(False, "NotKilled", ("?x",)),
     (True, "Alive", ("?x",))]
]


# -----------------------------------------
# STEP 2: DISPLAY FUNCTIONS
# -----------------------------------------

def display_literal(literal):
    sign, predicate, arguments = literal

    text = predicate + "(" + ", ".join(arguments) + ")"

    if not sign:
        return "NOT " + text

    return text


def display_clause(clause):

    if not clause:
        return "EMPTY CLAUSE (□)"

    return " OR ".join(
        display_literal(lit) for lit in clause
    )


print("========== KNOWLEDGE BASE ==========")

for i, clause in enumerate(KB, 1):
    print("C" + str(i) + ":", display_clause(clause))


# -----------------------------------------
# STEP 3: UNIFICATION
# -----------------------------------------

def is_variable(term):
    return term.startswith("?")


def dereference(term, substitution):

    while is_variable(term) and term in substitution:
        term = substitution[term]

    return term


def unify(term1, term2, substitution):

    term1 = dereference(term1, substitution)
    term2 = dereference(term2, substitution)

    if term1 == term2:
        return True

    if is_variable(term1):
        substitution[term1] = term2
        return True

    if is_variable(term2):
        substitution[term2] = term1
        return True

    return False


# -----------------------------------------
# STEP 4: APPLY SUBSTITUTION
# -----------------------------------------

def apply_substitution(clause, substitution):

    result = []

    for sign, predicate, arguments in clause:

        new_arguments = tuple(
            dereference(arg, substitution)
            for arg in arguments
        )

        result.append(
            (sign, predicate, new_arguments)
        )

    return result


# -----------------------------------------
# STEP 5: STANDARDIZE VARIABLES
# -----------------------------------------

counter = 0


def standardize(clause):

    global counter

    counter += 1

    mapping = {}
    result = []

    for sign, predicate, arguments in clause:

        new_arguments = []

        for arg in arguments:

            if is_variable(arg):

                if arg not in mapping:
                    mapping[arg] = arg + "_" + str(counter)

                new_arguments.append(mapping[arg])

            else:
                new_arguments.append(arg)

        result.append(
            (sign, predicate, tuple(new_arguments))
        )

    return result


# -----------------------------------------
# STEP 6: CANONICALIZE CLAUSES
# -----------------------------------------

def canonicalize(clause):

    mapping = {}
    result = []

    for sign, predicate, arguments in clause:

        new_arguments = []

        for arg in arguments:

            if is_variable(arg):

                if arg not in mapping:
                    mapping[arg] = "?v" + str(len(mapping))

                new_arguments.append(mapping[arg])

            else:
                new_arguments.append(arg)

        result.append(
            (sign, predicate, tuple(new_arguments))
        )

    return result


# -----------------------------------------
# STEP 7: RESOLUTION
# -----------------------------------------

def resolve(clause1, clause2):

    clause1 = standardize(clause1)
    clause2 = standardize(clause2)

    resolvents = []

    for i, lit1 in enumerate(clause1):

        for j, lit2 in enumerate(clause2):

            sign1, pred1, args1 = lit1
            sign2, pred2, args2 = lit2

            # Check complementary literals
            if pred1 != pred2 or sign1 == sign2:
                continue

            if len(args1) != len(args2):
                continue

            substitution = {}

            # Apply unification
            success = True

            for a, b in zip(args1, args2):

                if not unify(a, b, substitution):
                    success = False
                    break

            if not success:
                continue

            # Remove complementary literals
            remaining = (
                clause1[:i] + clause1[i+1:] +
                clause2[:j] + clause2[j+1:]
            )

            # Apply substitution
            new_clause = apply_substitution(
                remaining, substitution
            )

            # Remove duplicate literals
            new_clause = list(dict.fromkeys(new_clause))

            # Ignore tautological clauses
            literal_set = set(new_clause)

            tautology = any(
                (not sign, pred, args) in literal_set
                for sign, pred, args in new_clause
            )

            if not tautology:
                resolvents.append(
                    (new_clause, substitution)
                )

    return resolvents


# -----------------------------------------
# STEP 8: NEGATE THE QUERY
# -----------------------------------------

print("\n========== NEGATE THE QUERY ==========")

query = (True, "Likes", ("John", "Peanuts"))

negated_query = [
    (False, "Likes", ("John", "Peanuts"))
]

print("Original Query:", display_literal(query))
print("Negated Query:", display_clause(negated_query))

KB.append(negated_query)


# -----------------------------------------
# STEP 9: APPLY RESOLUTION ITERATIVELY
# -----------------------------------------

print("\n========== RESOLUTION PROOF ==========")

seen = set()

for clause in KB:
    canonical = canonicalize(clause)
    seen.add(tuple(canonical))

proved = False
step = 1
max_steps = 1000

while step <= max_steps:

    new_clauses = []

    for i in range(len(KB)):

        for j in range(i + 1, len(KB)):

            results = resolve(KB[i], KB[j])

            for new_clause, substitution in results:

                new_clause = canonicalize(new_clause)
                key = tuple(new_clause)

                if key in seen:
                    continue

                seen.add(key)

                print("\nResolution Step:", step)

                print("Clause 1:", display_clause(KB[i]))
                print("Clause 2:", display_clause(KB[j]))

                print("Unification:", substitution)

                print("New Clause:",
                      display_clause(new_clause))

                step += 1

                # Check for empty clause
                if len(new_clause) == 0:

                    proved = True
                    break

                new_clauses.append(new_clause)

            if proved:
                break

        if proved:
            break

    if proved or not new_clauses:
        break

    KB.extend(new_clauses)


# -----------------------------------------
# STEP 10: DISPLAY FINAL RESULT
# -----------------------------------------

print("\n========== FINAL RESULT ==========")

if proved:

    print("Empty Clause Obtained: □")
    print("Query is PROVED using Resolution!")
    print("\nTherefore, John likes peanuts.")

else:

    print("Empty clause not obtained.")
    print("Query could not be proved.")


========== KNOWLEDGE BASE ==========
C1: NOT Food(?x) OR Likes(John, ?x)
C2: Food(Apple)
C3: Food(Vegetables)
C4: NOT Eats(?x, ?y) OR NOT NotKilled(?x) OR Food(?y)
C5: Eats(Anil, Peanuts)
C6: Alive(Anil)
C7: NOT Eats(Anil, ?x) OR Eats(Harry, ?x)
C8: NOT Alive(?x) OR NotKilled(?x)
C9: NOT NotKilled(?x) OR Alive(?x)

========== NEGATE THE QUERY ==========
Original Query: Likes(John, Peanuts)
Negated Query: NOT Likes(John, Peanuts)

========== RESOLUTION PROOF ==========

Resolution Step: 1
Clause 1: NOT Food(?x) OR Likes(John, ?x)
Clause 2: Food(Apple)
Unification: {'?x_1': 'Apple'}
New Clause: Likes(John, Apple)

Resolution Step: 2
Clause 1: NOT Food(?x) OR Likes(John, ?x)
Clause 2: Food(Vegetables)
Unification: {'?x_3': 'Vegetables'}
New Clause: Likes(John, Vegetables)

Resolution Step: 3
Clause 1: NOT Food(?x) OR Likes(John, ?x)
Clause 2: NOT Eats(?x, ?y) OR NOT NotKilled(?x) OR Food(?y)
Unification: {'?x_5': '?y_6'}
New Clause: Likes(John, ?v0) OR NOT Eats(?v1, ?v0) OR NOT NotKilled(